# exp134_self_gr_multiscale_longtail_gate train

Posthoc train-side audit for exp090 self-GR multiscale signal as an auxiliary longtail / PF-dense disagreement gate confidence. This notebook does not train LightGBM and does not create a submission candidate.

## Contents

1. Setup and configuration
2. Input and gate contract
3. Run self-GR gate audit
4. Metrics and generated artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from self_gr_multiscale_longtail_gate import run_audit

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_ROWS_ENV = os.environ.get("EXPERIMENT_MAX_ROWS")
MAX_ROWS = int(MAX_ROWS_ENV) if MAX_ROWS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Output root:", paths.output_root)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max rows:", MAX_ROWS)

## 2. Input and gate contract

In [ ]:
contract = {
    "feature_cache": get_nested(config, "data.exp072_feature_cache"),
    "optional_exp073_predictions": get_nested(config, "data.exp073_predictions"),
    "optional_exp092_predictions": get_nested(config, "data.exp092_predictions"),
    "self_gr": get_nested(config, "self_gr"),
    "gate_variants": get_nested(config, "audit.gate_variants"),
    "expected_artifacts": get_nested(config, "audit.expected_train_artifacts"),
}
print(json.dumps(contract, indent=2, ensure_ascii=False))

## 3. Run self-GR gate audit

In [ ]:
summary = run_audit(config, paths, max_rows=MAX_ROWS)
paths.metrics_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True) + "\n")
print(json.dumps({
    "status": summary["status"],
    "rows": summary["rows"],
    "wells": summary["wells"],
    "best_prediction": summary["best_prediction"],
    "decision": summary["decision"],
}, indent=2, ensure_ascii=False))
print("Metrics written:", paths.metrics_path)

## 4. Metrics and generated artifacts

In [ ]:
artifact_paths = {key: Path(value) for key, value in summary["artifacts"].items()}
display(pd.DataFrame([{"artifact": key, "path": str(path), "exists": path.exists()} for key, path in artifact_paths.items()]))

metrics = pd.read_csv(artifact_paths["metrics"])
display(metrics.sort_values("rmse_tvt").head(20))

bucket_metrics = pd.read_csv(artifact_paths["bucket_metrics"])
display(bucket_metrics.head(20))

common_worst = pd.read_csv(artifact_paths["common_worst_metrics"])
display(common_worst.sort_values(["scope", "rmse_tvt"]).head(20))